# 09_very_high_batch_reconcile

Generated from `alteryx-workflows\09_very_high_batch_reconcile.yxmd` by compiler 2.2.0.

## How to use this notebook

1. Review and complete the source configuration.
2. Run cells from top to bottom.
3. Inspect the output previews and validate semantic parity.

> **Important:** Generated transformations are unvalidated. Confirm schemas, row counts, business rules, and destinations before promotion.


## Runtime imports

Imports and runtime helpers required by the generated PySpark transformations.


In [ ]:
from pyspark.sql import Window, functions as F


## Source ingestion

Loads and validates source datasets before transformation.


### Source ingestion 1 of 1

Produces `batch_a`, `batch_b`.


In [ ]:
batch_a = spark.createDataFrame([["C001", "West", "Online", "2026-01-12", "SKU-A-100", "1200", "4"], ["C002", "East", "Store", "2026-01-18", "SKU-B-220", "840", "2"], ["C001", "West", "Online", "2026-02-03", "SKU-A-140", "640", "1"], ["C003", "North", "Partner", "2026-02-11", "SKU-C-010", "2100", "6"], ["C004", "South", "Store", "2026-03-01", "SKU-B-220", "430", "3"], ["C002", "East", "Online", "2026-03-09", "SKU-A-100", "990", "2"], ["C005", "West", "Partner", "2026-03-21", "SKU-C-010", "150", "1"], ["C003", "North", "Store", "2026-04-02", "SKU-A-140", "1760", "5"]], ["CustomerID", "Region", "Channel", "OrderDate", "SKU", "Amount", "Units"])
batch_b = spark.createDataFrame([["C003", "North", "Store", "2026-04-02", "SKU-A-140", "1760", "5"], ["C005", "West", "Partner", "2026-03-21", "SKU-C-010", "150", "1"], ["C002", "East", "Online", "2026-03-09", "SKU-A-100", "990", "2"], ["C004", "South", "Store", "2026-03-01", "SKU-B-220", "430", "3"], ["C003", "North", "Partner", "2026-02-11", "SKU-C-010", "2100", "6"], ["C001", "West", "Online", "2026-02-03", "SKU-A-140", "640", "1"], ["C002", "East", "Store", "2026-01-18", "SKU-B-220", "840", "2"], ["C001", "West", "Online", "2026-01-12", "SKU-A-100", "1200", "4"]], ["CustomerID", "Region", "Channel", "OrderDate", "SKU", "Amount", "Units"])


## Transformations

Applies the translated Alteryx operations in dependency order.


### Transformations 1 of 2

Produces `batch_a_id`, `batch_b_id`, `region_key_a`, `region_key_b`, and 4 more.


In [ ]:
batch_a_id = batch_a.withColumn("RecordID", F.monotonically_increasing_id() + 1)
batch_b_id = batch_b.withColumn("RecordID", F.monotonically_increasing_id() + 1)
region_key_a = batch_a_id.withColumn("RegionKey", F.upper(F.col("Region")))
region_key_b = batch_b_id.withColumn("RegionKey", F.upper(F.col("Region")))
match_key_a = region_key_a.withColumn("MatchKey", F.concat_ws("|", F.col("CustomerID"), F.col("SKU"), F.col("OrderDate")))
match_key_b = region_key_b.withColumn("MatchKey", F.concat_ws("|", F.col("CustomerID"), F.col("SKU"), F.col("OrderDate")))
reconcile_batches = match_key_a.join(match_key_b, (match_key_a["MatchKey"] == match_key_b["MatchKey"]), 'inner').drop(match_key_b["MatchKey"])
amount_delta = reconcile_batches.withColumn("AmountDelta", F.col("Left_Amount").cast('double') - F.col("Right_Amount").cast('double'))


### Transformations 2 of 2

Produces `changed_amounts`, `delta_by_region`, `left_only_keys`, `right_only_keys`, and 3 more.


In [ ]:
changed_amounts = amount_delta.filter(F.col("AmountDelta") != "0")
delta_by_region = changed_amounts.groupBy(*["Left_Region"]).agg(F.sum(F.col("AmountDelta").cast('double')).alias("Sum_AmountDelta"), F.count(F.col("AmountDelta")).alias("RowCount"))
left_only_keys = reconcile_batches.dropDuplicates(["MatchKey"])
right_only_keys = reconcile_batches.dropDuplicates(["MatchKey"])
exceptions = left_only_keys.unionByName(right_only_keys, allowMissingColumns=True)
sort_exceptions = exceptions.orderBy(F.col("MatchKey").asc())
exception_sample = sort_exceptions.limit(20)


## Output preview

Displays terminal DataFrames for review. Replace preview-only output with approved destination writes after validation.


### Output preview 1 of 1


In [ ]:
display(delta_by_region)
display(exception_sample)
display(amount_delta)
